Responsibilities:

- star schema design
- 
- fact tables
- 
- dimension tables.

In [0]:
from pyspark.sql.functions import col

In [0]:
orders = spark.table("e_comm_databricks.pipeline_silver.orders")

order_items = spark.table("e_comm_databricks.pipeline_silver.order_items")

payments = spark.table("e_comm_databricks.pipeline_silver.order_payments")

customers = spark.table("e_comm_databricks.pipeline_silver.customers")

products = spark.table("e_comm_databricks.pipeline_silver.products")

sellers = spark.table("e_comm_databricks.pipeline_silver.sellers")


In [0]:
dim_customers = customers.select(

    "customer_id",
    "customer_city",
    "customer_state"

)


In [0]:
dim_customers.write \
 .format("delta") \
 .mode("overwrite") \
 .saveAsTable("e_comm_databricks.pipeline_gold.dim_customers")


In [0]:
dim_products = products.select("*")


In [0]:
dim_products.write \
 .format("delta") \
 .mode("overwrite") \
 .saveAsTable("e_comm_databricks.pipeline_gold.dim_products")


In [0]:
dim_sellers = sellers.select("*")


In [0]:
dim_sellers.write \
 .format("delta") \
 .mode("overwrite") \
 .saveAsTable("e_comm_databricks.pipeline_gold.dim_sellers")


In [0]:
fact_order_items = order_items.join(

    orders.select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp"
    ),

    on="order_id",

    how="left"

)


In [0]:
fact_order_items = fact_order_items.select(

    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "price",
    "freight_value",
    "customer_id",
    "order_status",
    "order_purchase_timestamp"

)


In [0]:
fact_order_items.write \
 .format("delta") \
 .mode("overwrite") \
 .saveAsTable("e_comm_databricks.pipeline_gold.fact_order_items")


In [0]:
fact_payments = payments.select(

    "order_id",
    "payment_sequential",
    "payment_type",
    "payment_installments",
    "payment_value"

)


In [0]:
fact_payments.write \
 .format("delta") \
 .mode("overwrite") \
 .saveAsTable("e_comm_databricks.pipeline_gold.fact_payments")
